
# DPO Pipeline for Aligning with AI Preference Data



In [1]:
import torch, os
from pathlib import Path
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          BitsAndBytesConfig, TrainingArguments)
from trl import DPOTrainer, DPOConfig
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
import pandas as pd, json

from peft import LoraConfig, PeftModel

In [2]:
RLAIF_DIR = "rlaif_data"
MODEL_ID_BASE = "mistralai/Mistral-7B-Instruct-v0.1"
FT_ADAPTER_DIR = "models/mistral-intruct_v01-sl"         
OUTPUT_DIR = "models"

SEED = 42

DATA_PATH = os.path.join(RLAIF_DIR,"gpt_preference_data_1000_selected_principle.json")


In [3]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

tok = AutoTokenizer.from_pretrained(MODEL_ID_BASE, use_fast=True)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID_BASE,
    quantization_config=bnb_config,
    device_map="auto",
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    records = json.load(f)

df = pd.DataFrame.from_records(records)
df = df[df["label"].isin(["A", "B"])]

df["chosen"]   = df.apply(lambda r: r["A"] if r["label"] == "A" else r["B"], axis=1)
df["rejected"] = df.apply(lambda r: r["B"] if r["label"] == "A" else r["A"], axis=1)

df = df[["prompt", "chosen", "rejected"]].dropna().reset_index(drop=True)
print("Loaded", len(df), "pairs")

train_df, val_df = train_test_split(df, test_size=0.1, random_state=SEED, shuffle=True)

ds = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df, preserve_index=False),
})

Loaded 978 pairs


In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID_BASE, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

policy_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID_BASE,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

policy_model = PeftModel.from_pretrained(policy_base, FT_ADAPTER_DIR, is_trainable=False)



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [9]:
dpo_args = DPOConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-5,
    max_steps=200,
    warmup_ratio=0.03,
    logging_steps=10,
    save_steps=100,
    lr_scheduler_type="cosine",
    bf16=True,
    remove_unused_columns=False,
    report_to=["none"],
    seed=SEED,
    beta=0.1,
    max_length=512,
    max_prompt_length=256,
)




In [12]:
trainer = DPOTrainer(
    model=policy_model,       
    args=dpo_args,              
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
)

trainer.train()

trainer.model.save_pretrained(f"{OUTPUT_DIR}/final_policy")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_policy")
print("Training complete and model saved.")


Extracting prompt in train dataset:   0%|          | 0/880 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/880 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/880 [00:00<?, ? examples/s]

Extracting prompt in eval dataset:   0%|          | 0/98 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/98 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/98 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss
10,0.694200
20,0.685900
30,0.692500
40,0.674100
50,0.677300
60,0.675600
70,0.694700
80,0.697400
90,0.689700
100,0.671500


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Training complete and model saved.
